# crypto-backtest — walkthrough

A short tour of the package: run the seeded end-to-end demo, look at the new
indicators (MACD / Bollinger / ATR), sweep fee & slippage assumptions, and
list the walk-forward validation windows.

**Honest framing.** The demo price path is a *seeded synthetic* zero-drift
random walk, so a costed crossover is expected to lose — the point is a
defensible, reproducible pipeline (no look-ahead; fees and slippage on every
trade), not a real-world edge.


In [ ]:
import sys

sys.path.insert(0, "../src")  # make the package importable from notebooks/

import backtest  # noqa: E402
from backtest.bars import resample_ohlcv  # noqa: E402
from backtest.demo import _BAR_RULE, _synthesize_ticks, run_demo  # noqa: E402
from backtest.indicators import atr, bollinger_bands, macd, sma  # noqa: E402
from backtest.performance import (  # noqa: E402
    calmar,
    exposure,
    sortino,
    turnover,
    win_rate,
)
from backtest.validation import sensitivity_sweep, walk_forward_splits  # noqa: E402

print("backtest", backtest.__version__)

## 1. Run the seeded demo (the real pipeline)

In [ ]:
result = run_demo(0)
result

## 2. Rebuild the demo bars

We reconstruct the exact bars the demo used so we can compute indicators and
sweep costs on them.


In [ ]:
ticks = _synthesize_ticks(0)
bars = resample_ohlcv(ticks, _BAR_RULE, ts="ts", price="price", size="size")
close = bars["close"].astype(float)
signal = (sma(close, 10) > sma(close, 30)).astype(float)
bars.tail()

## 3. New indicators: MACD, Bollinger bands, Wilder ATR

In [ ]:
macd_df = macd(close, fast=12, slow=26, signal=9)
bb = bollinger_bands(close, n=20, k=2.0)
atr14 = atr(bars["high"], bars["low"], close, n=14)

print("MACD (last 3 bars):")
print(macd_df.tail(3))
print("\nBollinger bands (last 3 bars):")
print(bb.tail(3))
print("\nATR-14 (last 3 bars):")
print(atr14.tail(3))

## 4. Cost sensitivity sweep

Re-run the *real* engine across a grid of fee and slippage assumptions. The
zero-cost row reproduces the gross backtest; total return falls as costs rise.


In [ ]:
sweep = sensitivity_sweep(
    close,
    signal,
    fee_grid=[0.0, 5.0, 10.0, 20.0],
    slippage_grid=[0.0, 5.0, 10.0],
)
sweep

## 5. Walk-forward validation windows

Non-overlapping out-of-sample test windows. Every test index sits strictly
after its training window, so a strategy fit on `train` is only reported on
data it never saw.


In [ ]:
n = len(close)
folds = list(walk_forward_splits(n, train=300, test=100))
print(f"{len(folds)} folds over {n} bars")
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train [{tr.start}:{tr.stop})  test [{te.start}:{te.stop})")

## 6. Extended performance stats

Sortino, Calmar, win rate, turnover, and exposure on the demo's own equity and
positions.


In [ ]:
equity = backtest.backtest(close, signal, fee_bps=10.0, slippage_bps=5.0)
rets = equity.pct_change().fillna(0.0)
positions = signal.shift(1).fillna(0.0)

print("sortino :", sortino(rets.to_numpy(), 525600))
print("calmar  :", calmar(equity.to_numpy(), 525600))
print("win_rate:", win_rate(rets.to_numpy()))
print("turnover:", turnover(positions.to_numpy()))
print("exposure:", exposure(positions.to_numpy()))